# Bayesian Gridding with Population Adjustment

This notebook demonstrates Bayesian spatial gridding of DHIS2 disease data after integrating WorldPop population data, followed by masking and visualization.

## Imports

In [ ]:
from io import StringIO

import pandas as pd
import geopandas as gpd

from masking import mask
from dhis2eo.data.worldpop import pop_total
from prepareDatawithPop import prepare_data_with_pop
from preparedata import prepare_data
from bayersianGrid import bayesian_grid
from plot import plotData

## 1. Data Preparation

In [ ]:
pe = "202501"  # Period

data = prepare_data(
    base_url="SomeURL",
    username="username",
    password="password",
    dx='jPEcKbn7jmh',
    pe=pe,
    ou_level="4"
)

# Read DHIS2 analytics into pandas
dataValues = pd.read_csv(StringIO(data))

dataValues.head()

## 2. Load WorldPop Population Data

In [ ]:
country_code = 'MWI'  # Malawi ISO code

pop_file = pop_total.get("2025", country_code)
pop_file

## 3. Prepare Data with Population

In [ ]:
# Combine disease data with population data
dataValues_ready = prepare_data_with_pop(dataValues, pop_file)

dataValues_ready.head()

## 4. Bayesian Gridding

In [ ]:
grd = bayesian_grid(dataValues_ready)
grd

## 5. Masking and Visualization

In [ ]:
districts_path = r"C:\\Users\\ShnkMn\\Documents\\CMS\\climate-tools\\docs\\data\\Districts.shp"

# Mask out water bodies / non-land areas
msk = mask(grd, districts_path)

# Load overlay for plotting
overlay = gpd.read_file(districts_path).to_crs(epsg=4326)

# Select first time slice for plotting
data_to_plot = msk.isel(time=0)

plotData(data_to_plot, overlay)